# Self-Attention by hand, then in numpy**Fundamentals of Large Language Models** — *Next-Word Prediction and Self-Attention*Vorlesung *Datensynthese und Sprachmodelle in der Medizin*This is the exercise from the **"Your turn"** slide. Do it on paper first — it is fourshort lines — then fill in the `TODO`s here and let the notebook check you.Only `numpy` is needed. Nothing is downloaded; everything runs offline.

## Setup — the toy from the lectureTwo tokens, two dimensions, the same learned matrices you saw on the slide.

In [ ]:
import numpy as npnp.set_printoptions(precision=3, suppress=True)# ---- the toy from the lecture -------------------------------------------# two tokens of the same sentence, two dimensions so you can do it by hande1 = np.array([1.0, 2.0])      # token 1e2 = np.array([0.0, 2.0])      # token 2   <- the one on the exercise slideq  = np.array([2.0, 2.0])      # the query we are asking withW_K = np.array([[0.0, 1.0],    # the learned key matrix                [1.0, 0.0]])W_V = np.array([[1.0,  1.0],   # the learned value matrix                [1.0, -1.0]])d_k = 2                        # the head width, so the scaling is sqrt(2)

## Part A — the four lines from the slide> **GIVEN** `q = (2, 2)`,  `W_K = [[0,1],[1,0]]`,  `W_V = [[1,1],[1,-1]]`,  `e2 = (0, 2)`>> **FIND**, in order: `k2`  →  `s`  →  `alpha`  →  `delta_e`### TODO 1 — the keysEach token advertises itself with a key, `k = W_K e`.

In [ ]:
k1 = W_K @ e1          # done for you - token 1k2 = None              # TODO 1: the key for token 2print("k1 =", k1)print("k2 =", k2)

### TODO 2 — the scoresHow well does the query match each key? A dot product, divided by `sqrt(d_k)` so thesize of the score does not grow with the width of the head.$$s_{i} = \frac{q \cdot k_i}{\sqrt{d_k}}$$

In [ ]:
# TODO 2: build the array of the two scaled scores, in order [s1, s2]s = Noneprint("s =", s)

### TODO 3 — softmaxTurn the two scores into attention weights that are positive and sum to exactly 1.$$\alpha_i = \frac{\exp(s_i)}{\sum_j \exp(s_j)}$$*Hint: subtract `s.max()` before exponentiating. It changes nothing mathematically andit stops `exp` overflowing — every real implementation does this.*

In [ ]:
def softmax(x):    # TODO 3a: return the softmax of the 1-D array x    return Nonealpha = None           # TODO 3b: apply it to sprint("alpha =", alpha)print("sums to", None if alpha is None else alpha.sum())

### TODO 4 — the values, and the updateAttention does not output a weight. It outputs a **vector**: the value vectors mixed inthe proportions `alpha`.$$v_i = W_V e_i \qquad \Delta e = \sum_i \alpha_i \, v_i$$

In [ ]:
v1 = None              # TODO 4av2 = None              # TODO 4bdelta_e = None         # TODO 4c: the weighted sumprint("v1 =", v1)print("v2 =", v2)print("delta_e =", delta_e)

### Check yourselfRun this once you have filled in all four.

In [ ]:
def check(name, got, want, tol=1e-2):    if got is None:        print(f"  ----   {name:<14} not filled in yet")        return False    got = np.atleast_1d(np.asarray(got, dtype=float))    want = np.atleast_1d(np.asarray(want, dtype=float))    ok = got.shape == want.shape and np.allclose(got, want, atol=tol)    print(("  OK   " if ok else "  WRONG") + f"  {name:<14} "          f"you: {np.round(got, 3)}   expected: {np.round(want, 3)}")    return okprint("Part A")results = [    check("k2",    k2,    [2.0, 0.0]),    check("s",     s,     [4.243, 2.828]),    check("alpha", alpha, [0.805, 0.195]),    check("v1",    v1,    [3.0, -1.0]),    check("v2",    v2,    [2.0, -2.0]),    check("delta", delta_e, [2.805, -1.195]),]print()if all(results):    print("all correct")else:    print("not finished - fill in the lines marked ---- and fix any marked WRONG")

**What the answer means.** Token 2 gets about 20 % of the attention and token 1 about80 %, even though both are in the same sentence. Token 2's key points away from thequery — and attention is a *similarity*, not a distance.

## Part B — the whole sentenceSame arithmetic, eight tokens instead of two. These are the scores from the grid in thelecture.

In [ ]:
TOKENS = ["Known", "allergy", "to", "penicillin", ";",          "treated", "with", "ceftriaxone"]# The scaled scores s[t, i] for the sentence, exactly as on the lecture's grid# (rows = the token asking, columns = the token answering).S = np.array([    [ 1.2, -0.4, -0.4, -0.4, -0.4, -0.4, -0.4, -0.4],    [ 1.1,  1.2, -0.4,  2.6, -0.4, -0.4, -0.4, -0.4],    [-0.4,  0.9,  1.2, -0.4, -0.4, -0.4, -0.4, -0.4],    [ 0.7,  2.2, -0.4,  1.2, -0.4, -0.4, -0.4, -0.4],    [-0.4, -0.4, -0.4, -0.4,  1.2, -0.4, -0.4, -0.4],    [-0.4,  1.3, -0.4,  1.1, -0.4,  1.2, -0.4, -0.4],    [-0.4, -0.4, -0.4, -0.4, -0.4,  1.4,  1.2, -0.4],    [-0.4,  2.4, -0.4,  2.9, -0.4,  1.2,  0.6,  1.2],])

### TODO 5 — mask, then normaliseTwo things to do:1. **Mask** — a token may not look at tokens that come *after* it. Set those scores to   `-inf` **before** the softmax, so they become exactly zero after it.2. **Softmax each row**, so every row is a probability distribution over the tokens so   far.

In [ ]:
def attention_pattern(S):    """S -> the masked, row-normalised attention pattern."""    n = S.shape[0]    M = S.copy()    # TODO 5a: set the strictly-upper triangle of M to -np.inf    #          (hint: np.triu_indices(n, 1))    # TODO 5b: softmax each ROW of M and return it    return NoneA = attention_pattern(S)A

In [ ]:
if A is not None:    t = TOKENS.index("ceftriaxone")    print('query = "ceftriaxone"\n')    for tok, w in zip(TOKENS, A[t]):        bar = "#" * int(round(w * 40))        print(f"  {tok:<12} {w:5.1%}  {bar}")    print(f"\n  row sums to {A[t].sum():.3f}")    print("\nExpected from the lecture: penicillin 46 %, allergy 28 %.")

**The point of the whole lecture, in one number.** The query is *"ceftriaxone"*, andalmost half its attention lands on *"penicillin"* — nine tokens back. That is the reachan n-gram does not have.

## Part C — from prediction to synthesis *(optional)*The closing slide: the same machinery run forward writes text. Here is the wholegeneration loop, in eight lines.`temperature` is the slider: divide the scores by `T` before the softmax. Low `T`repeats, high `T` invents.

In [ ]:
VOCAB  = ["temperature", "pulse", "weight"]LOGITS = np.array([2.0, 1.0, 0.0])      # the toy from the temperature slidedef sample_once(logits, T=1.0, rng=None):    rng = rng or np.random.default_rng(0)    p = np.exp((logits - logits.max()) / T)    p = p / p.sum()    return VOCAB[rng.choice(len(VOCAB), p=p)], pfor T in (0.3, 1.0, 2.5):    rng = np.random.default_rng(7)    draws = [sample_once(LOGITS, T, rng)[0] for _ in range(12)]    _, p = sample_once(LOGITS, T)    print(f"T = {T:>3}   p = {np.round(p, 2)}   ->  " + " ".join(draws[:6]))

**Try it:** turn `T` up and run the cell again. The same model, the same prompt — adifferent note every time. That is data synthesis, and it is also why a generated notehas to be *checked* rather than trusted.